# Uber Pickups — NYC Hot Zones
## Unsupervised Machine Learning Project
Algorithmes utilisés : **KMeans** et **DBSCAN**  
Données : Uber Trip Data — Avril à Septembre 2014 + Janvier–Juin 2015 (New York City)

## 1. Import des librairies

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer

import warnings
warnings.filterwarnings('ignore')

## 2. Exploration d'un fichier unique — `uber-raw-data-apr14.csv`
On commence par analyser un seul fichier pour comprendre la structure des données.

In [ ]:
dataset = pd.read_csv('uber-raw-data-apr14.csv')

print('Shape :', dataset.shape)
print()
print('Colonnes :', dataset.columns.tolist())
print()
print('Statistiques globales')
print(dataset.describe(include='all'))
print()
print('Valeurs manquantes (%)')
print(dataset.isnull().sum() / dataset.shape[0])

In [ ]:
# Conversion de la colonne Date/Time en datetime
dataset['Date_Time_date_time'] = pd.to_datetime(dataset['Date/Time'])

dataset['year']    = dataset['Date_Time_date_time'].dt.year
dataset['month']   = dataset['Date_Time_date_time'].dt.month
dataset['day']     = dataset['Date_Time_date_time'].dt.day
dataset['weekday'] = dataset['Date_Time_date_time'].dt.day_name()
dataset['hour']    = dataset['Date_Time_date_time'].dt.hour
dataset['minute']  = dataset['Date_Time_date_time'].dt.minute

dataset = dataset.drop(['Date_Time_date_time'], axis=1)
dataset.head()

In [ ]:
# Distribution des pickups par heure
pickups_by_hour = dataset.groupby('hour').size().reset_index(name='count')

fig = px.bar(
    pickups_by_hour,
    x='hour', y='count',
    title='Pickups par heure — Avril 2014',
    labels={'hour': 'Heure', 'count': 'Nombre de pickups'}
)
fig.show()

In [ ]:
# Distribution des pickups par jour de la semaine
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

pickups_by_day = dataset.groupby('weekday').size().reset_index(name='count')
pickups_by_day['weekday'] = pd.Categorical(pickups_by_day['weekday'], categories=days_order, ordered=True)
pickups_by_day = pickups_by_day.sort_values('weekday')

fig = px.bar(
    pickups_by_day,
    x='weekday', y='count',
    title='Pickups par jour de la semaine — Avril 2014',
    labels={'weekday': 'Jour', 'count': 'Nombre de pickups'}
)
fig.show()

In [ ]:
# Carte générale — sample pour lisibilité
sample = dataset.sample(n=10000, random_state=42)

fig = px.scatter_map(
    data_frame=sample,
    lat='Lat', lon='Lon',
    color='hour',
    title='Aperçu général des pickups — NYC Avril 2014 (sample 10 000)',
    zoom=10
)
fig.show()

## 3. KMeans — Analyse sur un snapshot précis (day=1, hour=10h)
On commence petit pour calibrer l'algorithme avant de généraliser.

In [ ]:
dataset_day1     = dataset.loc[dataset['day'] == 1, :]
dataset_day1_10h = dataset_day1.loc[dataset_day1['hour'] == 10, :].copy()

print(f'Pickups sur ce snapshot : {len(dataset_day1_10h)}')

In [ ]:
sc = StandardScaler()
X_snap_scaled = sc.fit_transform(dataset_day1_10h[['Lat', 'Lon']])

# Méthode du coude avec Yellowbrick
model = KMeans(random_state=0, n_init='auto')
viz = KElbowVisualizer(model, k=(2, 11), timings=False)
viz.fit(X_snap_scaled)
viz.show()

optimal_k_snap = viz.elbow_value_
print(f'Nombre optimal de clusters (Elbow) : {optimal_k_snap}')

In [ ]:
# Score de silhouette
model_sil = KMeans(n_clusters=optimal_k_snap, random_state=0, n_init='auto')
viz_sil = SilhouetteVisualizer(model_sil)
viz_sil.fit(X_snap_scaled)
viz_sil.show()

In [ ]:
# Application KMeans
kmeans_snap = KMeans(n_clusters=optimal_k_snap, random_state=0, n_init='auto')
dataset_day1_10h['Cluster_KMeans'] = kmeans_snap.fit_predict(X_snap_scaled)

fig = px.scatter_map(
    data_frame=dataset_day1_10h,
    lat='Lat', lon='Lon',
    color='Cluster_KMeans',
    title=f'KMeans (k={optimal_k_snap}) — Day 1, 10h',
    zoom=10
)
fig.show()

## 4. DBSCAN — Même snapshot + Comparaison avec KMeans

In [ ]:
db_snap = DBSCAN(eps=0.1, min_samples=4, metric='manhattan', algorithm='brute')
dataset_day1_10h['Cluster_DBSCAN'] = db_snap.fit_predict(X_snap_scaled)

n_clusters_db = len(set(db_snap.labels_)) - (1 if -1 in db_snap.labels_ else 0)
n_noise       = list(db_snap.labels_).count(-1)
print(f'Clusters détectés : {n_clusters_db}')
print(f'Points bruit : {n_noise} ({n_noise / len(db_snap.labels_) * 100:.1f}%)')

In [ ]:
# Carte KMeans
fig_km = px.scatter_map(
    data_frame=dataset_day1_10h,
    lat='Lat', lon='Lon',
    color='Cluster_KMeans',
    title=f'KMeans (k={optimal_k_snap}) — Day 1, 10h',
    zoom=10
)
fig_km.show()

# Carte DBSCAN (sans les points bruit)
fig_db = px.scatter_map(
    data_frame=dataset_day1_10h.loc[dataset_day1_10h['Cluster_DBSCAN'] != -1, :],
    lat='Lat', lon='Lon',
    color='Cluster_DBSCAN',
    title=f'DBSCAN ({n_clusters_db} clusters) — Day 1, 10h (bruit exclu)',
    zoom=10
)
fig_db.show()

**Comparaison KMeans vs DBSCAN :**

| Critère | KMeans | DBSCAN |
|---|---|---|
| Nombre de clusters | Fixé par k (Elbow) | Automatique |
| Gestion du bruit | Non | Oui (label = -1) |
| Forme des clusters | Sphérique | Toute forme |
| Paramètres | k | eps, min_samples |
| Usage Uber | Zones prédéfinies | Hotspots organiques |

## 5. KMeans — Généralisation heure par heure (day=1)

In [ ]:
sc = StandardScaler()
n_clusters = 6
kmeans_day = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')

center_list = []

for hour in sorted(dataset_day1['hour'].unique()):
    df_hour = dataset_day1.loc[dataset_day1['hour'] == hour, :]
    X_hour_scaled = sc.fit_transform(df_hour[['Lat', 'Lon']])

    kmeans_day.fit(X_hour_scaled)
    centroids = sc.inverse_transform(kmeans_day.cluster_centers_)

    for cluster in range(n_clusters):
        center_list.append({
            'Lat': centroids[cluster][0],
            'Lon': centroids[cluster][1],
            'hour': hour,
            'cluster': cluster
        })

centers_day1 = pd.DataFrame(center_list)

fig = px.scatter_map(
    data_frame=centers_day1,
    lat='Lat', lon='Lon',
    color='cluster',
    animation_frame='hour',
    title='Hot Zones KMeans — Day 1, heure par heure',
    zoom=10
)
fig.show()

## 6. KMeans — Généralisation par jour de la semaine (fichier unique)

In [ ]:
sc = StandardScaler()
n_clusters = 6
kmeans_wd = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

weekday_centers_list = []

for weekday in days_order:
    df_wd = dataset.loc[dataset['weekday'] == weekday, :]

    for hour in sorted(df_wd['hour'].unique()):
        df_wd_hour = df_wd.loc[df_wd['hour'] == hour, :]

        if len(df_wd_hour) < n_clusters:  # Pas assez de points
            continue

        X_scaled = sc.fit_transform(df_wd_hour[['Lat', 'Lon']])
        kmeans_wd.fit(X_scaled)
        centroids = sc.inverse_transform(kmeans_wd.cluster_centers_)

        for cluster in range(n_clusters):
            weekday_centers_list.append({
                'Lat': centroids[cluster][0],
                'Lon': centroids[cluster][1],
                'weekday': weekday,
                'hour': hour,
                'cluster': cluster
            })

weekday_centers = pd.DataFrame(weekday_centers_list)
weekday_centers['weekday'] = pd.Categorical(weekday_centers['weekday'], categories=days_order, ordered=True)
weekday_centers = weekday_centers.sort_values(['weekday', 'hour'])

fig = px.scatter_map(
    data_frame=weekday_centers,
    lat='Lat', lon='Lon',
    color='cluster',
    animation_frame='weekday',
    title='Hot Zones Uber par jour de la semaine — Avril 2014',
    zoom=10
)
fig.show()

---
## 7. Chargement de tous les fichiers CSV

Les fichiers disponibles couvrent deux périodes avec des **structures de colonnes différentes** :

| Fichiers | Période | Colonne date | Colonne base |
|---|---|---|---|
| `uber-raw-data-apr14.csv` à `uber-raw-data-sep14.csv` | Avr–Sep 2014 | `Date/Time` | `Base` |
| `uber-raw-data-janjune-15.csv` | Jan–Jun 2015 | `Pickup_date` | `Dispatching_base_num` |

On normalise les deux formats pour obtenir un seul DataFrame unifié.

In [ ]:
# Liste des fichiers 2014 (format A)
files_2014 = [
    'uber-raw-data-apr14.csv',
    'uber-raw-data-may14.csv',
    'uber-raw-data-jun14.csv',
    'uber-raw-data-jul14.csv',
    'uber-raw-data-aug14.csv',
    'uber-raw-data-sep14.csv',
]

# Fichier 2015 (format B — colonnes différentes)
file_2015 = 'uber-raw-data-janjune-15.csv'

# ── Chargement des fichiers 2014 ──────────────────────────────────────────
dfs_2014 = []

for filepath in files_2014:
    df = pd.read_csv(filepath)

    # Normalisation des colonnes vers un format commun
    df = df.rename(columns={'Date/Time': 'datetime_raw', 'Base': 'base'})
    df['source_file'] = os.path.basename(filepath)

    dfs_2014.append(df)
    print(f'{filepath} → {df.shape[0]:,} lignes | colonnes : {df.columns.tolist()}')

df_2014 = pd.concat(dfs_2014, ignore_index=True)
print(f'\nTotal 2014 : {df_2014.shape[0]:,} lignes')

In [ ]:
# ── Chargement du fichier 2015 ────────────────────────────────────────────
df_2015_raw = pd.read_csv(file_2015)

print('Colonnes fichier 2015 :', df_2015_raw.columns.tolist())
print('Shape :', df_2015_raw.shape)
df_2015_raw.head()

In [ ]:
# Normalisation du fichier 2015 vers le même format que 2014
df_2015 = df_2015_raw.rename(columns={
    'Pickup_date':            'datetime_raw',
    'Dispatching_base_num':   'base'
})

# On conserve uniquement les colonnes communes
df_2015 = df_2015[['datetime_raw', 'Lat', 'Lon', 'base']].copy()
df_2015['source_file'] = os.path.basename(file_2015)

print(f'Fichier 2015 normalisé : {df_2015.shape[0]:,} lignes')
df_2015.head()

In [ ]:
# ── Fusion des deux périodes ───────────────────────────────────────────────
df_all = pd.concat([df_2014, df_2015], ignore_index=True)

# Conversion datetime
df_all['datetime_raw'] = pd.to_datetime(df_all['datetime_raw'])

# Feature engineering temporel
df_all['year']    = df_all['datetime_raw'].dt.year
df_all['month']   = df_all['datetime_raw'].dt.month
df_all['day']     = df_all['datetime_raw'].dt.day
df_all['weekday'] = df_all['datetime_raw'].dt.day_name()
df_all['hour']    = df_all['datetime_raw'].dt.hour
df_all['minute']  = df_all['datetime_raw'].dt.minute

df_all = df_all.drop(columns=['datetime_raw'])

print('Shape final :', df_all.shape)
print()
print('Répartition par fichier source :')
print(df_all.groupby('source_file').size().sort_index())
print()
print('Valeurs manquantes :')
print(df_all.isnull().sum())

In [ ]:
df_all.head()

## 8. Exploration globale — tous les fichiers
On analyse maintenant les patterns sur l'ensemble de la période (Avril 2014 – Juin 2015).

In [ ]:
# Pickups par mois
df_all['year_month'] = df_all['year'].astype(str) + '-' + df_all['month'].astype(str).str.zfill(2)
pickups_by_month = df_all.groupby('year_month').size().reset_index(name='count').sort_values('year_month')

fig = px.bar(
    pickups_by_month,
    x='year_month', y='count',
    title='Nombre de pickups par mois — Tous fichiers',
    labels={'year_month': 'Mois', 'count': 'Nombre de pickups'}
)
fig.show()

In [ ]:
# Pickups par heure — tous fichiers
pickups_by_hour_all = df_all.groupby('hour').size().reset_index(name='count')

fig = px.bar(
    pickups_by_hour_all,
    x='hour', y='count',
    title='Pickups par heure — Tous fichiers',
    labels={'hour': 'Heure', 'count': 'Nombre de pickups'}
)
fig.show()

In [ ]:
# Pickups par jour de la semaine — tous fichiers
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

pickups_by_day_all = df_all.groupby('weekday').size().reset_index(name='count')
pickups_by_day_all['weekday'] = pd.Categorical(pickups_by_day_all['weekday'], categories=days_order, ordered=True)
pickups_by_day_all = pickups_by_day_all.sort_values('weekday')

fig = px.bar(
    pickups_by_day_all,
    x='weekday', y='count',
    title='Pickups par jour de la semaine — Tous fichiers',
    labels={'weekday': 'Jour', 'count': 'Nombre de pickups'}
)
fig.show()

In [ ]:
# Carte générale — sample sur tous les fichiers
sample_all = df_all.sample(n=20000, random_state=42)

fig = px.scatter_map(
    data_frame=sample_all,
    lat='Lat', lon='Lon',
    color='hour',
    title='Aperçu général — Tous pickups NYC (sample 20 000)',
    zoom=10
)
fig.show()

## 9. KMeans — Hot zones par jour de la semaine (tous fichiers)
On regroupe par weekday sur l'ensemble de la période pour avoir des patterns robustes.

In [ ]:
sc = StandardScaler()
n_clusters = 6
kmeans_all = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
days_order  = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

all_centers_list = []

for weekday in days_order:
    df_wd = df_all.loc[df_all['weekday'] == weekday, :]

    for hour in sorted(df_wd['hour'].unique()):
        df_wd_hour = df_wd.loc[df_wd['hour'] == hour, :]

        if len(df_wd_hour) < n_clusters:
            continue

        X_scaled = sc.fit_transform(df_wd_hour[['Lat', 'Lon']])
        kmeans_all.fit(X_scaled)
        centroids = sc.inverse_transform(kmeans_all.cluster_centers_)

        for cluster in range(n_clusters):
            all_centers_list.append({
                'Lat':     centroids[cluster][0],
                'Lon':     centroids[cluster][1],
                'weekday': weekday,
                'hour':    hour,
                'cluster': cluster
            })

all_centers = pd.DataFrame(all_centers_list)
all_centers['weekday'] = pd.Categorical(all_centers['weekday'], categories=days_order, ordered=True)
all_centers = all_centers.sort_values(['weekday', 'hour'])

print(f'Centroïdes calculés : {len(all_centers)}')

In [ ]:
# Carte animée par jour de la semaine — tous fichiers
fig = px.scatter_map(
    data_frame=all_centers,
    lat='Lat', lon='Lon',
    color='cluster',
    animation_frame='weekday',
    title='Hot Zones Uber — par jour de la semaine (Avr 2014 – Jun 2015)',
    zoom=10
)
fig.show()

In [ ]:
# Carte animée par heure pour le vendredi — tous fichiers
df_friday_all = all_centers.loc[all_centers['weekday'] == 'Friday', :]

fig = px.scatter_map(
    data_frame=df_friday_all,
    lat='Lat', lon='Lon',
    color='cluster',
    animation_frame='hour',
    title='Hot Zones — Vendredi, heure par heure (tous fichiers)',
    zoom=10
)
fig.show()

## 10. DBSCAN — Tous fichiers, snapshot Vendredi 18h
On applique DBSCAN sur un snapshot représentatif (heure de pointe du soir en semaine) agrégé sur tous les vendredis.

In [ ]:
# Tous les vendredis à 18h sur l'ensemble de la période
df_fri_18h = df_all.loc[
    (df_all['weekday'] == 'Friday') & (df_all['hour'] == 18), :
].copy()

print(f'Pickups (Vendredi 18h, tous fichiers) : {len(df_fri_18h):,}')

sc_fri = StandardScaler()
X_fri_scaled = sc_fri.fit_transform(df_fri_18h[['Lat', 'Lon']])

# KMeans
kmeans_fri = KMeans(n_clusters=6, random_state=0, n_init='auto')
df_fri_18h['Cluster_KMeans'] = kmeans_fri.fit_predict(X_fri_scaled)

# DBSCAN
db_fri = DBSCAN(eps=0.1, min_samples=4, metric='manhattan', algorithm='brute')
df_fri_18h['Cluster_DBSCAN'] = db_fri.fit_predict(X_fri_scaled)

n_db = len(set(db_fri.labels_)) - (1 if -1 in db_fri.labels_ else 0)
n_noise = list(db_fri.labels_).count(-1)
print(f'DBSCAN — Clusters : {n_db} | Bruit : {n_noise} ({n_noise/len(db_fri.labels_)*100:.1f}%)')

In [ ]:
# Carte KMeans — Vendredi 18h (tous fichiers)
fig_km = px.scatter_map(
    data_frame=df_fri_18h,
    lat='Lat', lon='Lon',
    color='Cluster_KMeans',
    title='KMeans (k=6) — Vendredi 18h (tous fichiers)',
    zoom=10
)
fig_km.show()

# Carte DBSCAN — Vendredi 18h (tous fichiers)
fig_db = px.scatter_map(
    data_frame=df_fri_18h.loc[df_fri_18h['Cluster_DBSCAN'] != -1, :],
    lat='Lat', lon='Lon',
    color='Cluster_DBSCAN',
    title=f'DBSCAN ({n_db} clusters) — Vendredi 18h (tous fichiers, bruit exclu)',
    zoom=10
)
fig_db.show()

## 11. Conclusion

Ce projet a permis d'identifier les **hot zones** de pickups Uber à New York City sur une période de 14 mois (Avril 2014 – Juin 2015) via deux approches non supervisées :

- **KMeans** : rapide, interprétable, k zones cibles fixes. L'Elbow method et le score de silhouette guident le choix de k.

- **DBSCAN** : découverte automatique des clusters, gestion du bruit. Révèle des concentrations naturelles sans contraindre leur forme ni leur nombre.

**Résultats clés :**
- La demande croît significativement entre 2014 et 2015
- Les heures de pointe (7h–9h, 17h–19h) concentrent la majorité des pickups
- Les vendredis et samedis soirs présentent les plus fortes densités
- Midtown Manhattan reste la zone dominante quelle que soit la période

> Ces hot zones pourraient être intégrées dans l'app Uber pour guider les chauffeurs en temps réel.